[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hankpark0706/OL7014/blob/main/notebooks/week6_lap.ipynb)

In [ ]:
%pip install -q gurobipy

In [ ]:
import gurobipy as gp
from gurobipy import GRB

params = {
    "WLSACCESSID": "paste your ACCESSID here",
    "WLSSECRET":   "paste your SECRET here",
    "LICENSEID":   123456,   # the number itself, no quotes
}
env = gp.Env(params=params)

# LAP --- the game's five nodes

![the network](https://raw.githubusercontent.com/hankpark0706/OL7014/main/notebooks/img/week6_lap_network.png)

$$
\begin{aligned}
\min \quad & 100 \textstyle\sum_{j} x_j \;+\; 10 \sum_{j} r_j \;+\; \sum_{(i,j)} t_{ij}\, y_{ij} \;+\; 60 \sum_{j} q_j \;+\; \sum_{j} s_j \\
\text{s.t.} \quad & \textstyle\sum_{j} y_{ij} - \sum_{j} y_{ji} = r_i - d_i + q_i - s_i \quad (\forall i) \\
& \textstyle\sum_{j} r_j \le 55, \qquad r_j \le 55\, x_j, \qquad \sum_{j} x_j \le 2 \\
& y, r, s \ge 0, \quad 0 \le q_j \le d_j, \quad x_j \in \{0,1\}
\end{aligned}
$$

Each road is two directed arcs: $y_{AC}$ and $y_{CA}$ both exist, same cost.

## Live coding --- fill in the blanks

Skeleton for building the model live in class, one comment at a time.

In [ ]:
# Initialize the model gp.Model()


# switches: xA, ..., xE binary --- every node is a candidate site


# reserves rA, ..., rE and the ten arc flows yAC, yCA, yBC, yCB, yCD, yDC, yCE, yEC, yDE, yED


# shortfalls qA (ub=12), ..., qE (ub=24) and surpluses sA, ..., sE


# objective: open + hold + ship + fail + waste --- lap.setObjective(..., GRB.MINIMIZE)


# conservation, one row per node --- keep handles: nodeA = lap.addConstr(...)


# budget, linking, at most p = 2 facilities


# Model is set up. Let's optimize --- lap.optimize()


# Let's print the plan: which sites open, the reserves, the flows, the shortages, the cost

## Reference: the completed model

By hand --- no dictionaries, no loops, no `quicksum` --- so every term matches the math above.

In [ ]:
lap = gp.Model("lap")

# switches -- one per node, every node a candidate
xA = lap.addVar(vtype=GRB.BINARY, name="xA")
xB = lap.addVar(vtype=GRB.BINARY, name="xB")
xC = lap.addVar(vtype=GRB.BINARY, name="xC")
xD = lap.addVar(vtype=GRB.BINARY, name="xD")
xE = lap.addVar(vtype=GRB.BINARY, name="xE")

# reserves -- tons are divisible, so continuous (lower bound 0 by default)
rA = lap.addVar(name="rA")
rB = lap.addVar(name="rB")
rC = lap.addVar(name="rC")
rD = lap.addVar(name="rD")
rE = lap.addVar(name="rE")

# flows -- each road twice, once per direction
yAC = lap.addVar(name="yAC"); yCA = lap.addVar(name="yCA")
yBC = lap.addVar(name="yBC"); yCB = lap.addVar(name="yCB")
yCD = lap.addVar(name="yCD"); yDC = lap.addVar(name="yDC")
yCE = lap.addVar(name="yCE"); yEC = lap.addVar(name="yEC")
yDE = lap.addVar(name="yDE"); yED = lap.addVar(name="yED")

# shortfalls (capped by demand) and surpluses
qA = lap.addVar(ub=12, name="qA"); sA = lap.addVar(name="sA")
qB = lap.addVar(ub=20, name="qB"); sB = lap.addVar(name="sB")
qC = lap.addVar(ub=8,  name="qC"); sC = lap.addVar(name="sC")
qD = lap.addVar(ub=16, name="qD"); sD = lap.addVar(name="sD")
qE = lap.addVar(ub=24, name="qE"); sE = lap.addVar(name="sE")

# objective -- open + hold + ship + fail + waste
lap.setObjective(
    100 * (xA + xB + xC + xD + xE)
    + 10 * (rA + rB + rC + rD + rE)
    + 4.5 * (yAC + yCA) + 4.9 * (yBC + yCB) + 5.1 * (yCD + yDC)
    + 5.2 * (yCE + yEC) + 4.0 * (yDE + yED)
    + 60 * (qA + qB + qC + qD + qE)
    + (sA + sB + sC + sD + sE),
    GRB.MINIMIZE)

# conservation -- MCNF's row at every node: outflow - inflow = r - d + q - s
nodeA = lap.addConstr(yAC - yCA == rA - 12 + qA - sA, name="nodeA")
nodeB = lap.addConstr(yBC - yCB == rB - 20 + qB - sB, name="nodeB")
nodeC = lap.addConstr(yCA + yCB + yCD + yCE - yAC - yBC - yDC - yEC == rC - 8 + qC - sC, name="nodeC")
nodeD = lap.addConstr(yDC + yDE - yCD - yED == rD - 16 + qD - sD, name="nodeD")
nodeE = lap.addConstr(yEC + yED - yCE - yDE == rE - 24 + qE - sE, name="nodeE")

# budget, linking (M = B = 55), at most p = 2 facilities
lap.addConstr(rA + rB + rC + rD + rE <= 55, name="budget")
lap.addConstr(rA <= 55 * xA, name="linkA")
lap.addConstr(rB <= 55 * xB, name="linkB")
lap.addConstr(rC <= 55 * xC, name="linkC")
lap.addConstr(rD <= 55 * xD, name="linkD")
lap.addConstr(rE <= 55 * xE, name="linkE")
lap.addConstr(xA + xB + xC + xD + xE <= 2, name="cardinality")

lap.optimize()

print(f"\nopen: A={round(xA.X)} B={round(xB.X)} C={round(xC.X)} D={round(xD.X)} E={round(xE.X)}")
print(f"reserves: rB = {rB.X:g}, rE = {rE.X:g}")
print(f"flows: E->D = {yED.X:g}   shortages: qA = {qA.X:g}, qC = {qC.X:g}, qD = {qD.X:g}")
print(f"total cost = {lap.ObjVal:g}   (the game's do-nothing plan costs 4800)")

## The same model, written for any network

The reference model spells out every term so each one matches the math. For a real network, write the **data** as dictionaries and let Gurobi build the rows: `y.sum(i, "*")` sums the flow **out of** node $i$ and `y.sum("*", i)` the flow **into** it, so one `addConstrs` line writes the conservation row at every node --- 5 nodes or 500. Same model, same answer.

In [ ]:
d = {"A": 12, "B": 20, "C": 8, "D": 16, "E": 24}                       # demand
roads = {("A", "C"): 4.5, ("B", "C"): 4.9, ("C", "D"): 5.1, ("C", "E"): 5.2, ("D", "E"): 4.0}
t = {**roads, **{(j, i): c for (i, j), c in roads.items()}}           # each road, both ways
N, B, p = list(d), 55, 2

lap2 = gp.Model("lap2")
x = lap2.addVars(N, vtype=GRB.BINARY, name="x")
r = lap2.addVars(N, name="r")
y = lap2.addVars(t.keys(), name="y")          # one flow per directed arc
q = lap2.addVars(N, ub=d, name="q")            # shortfall capped by demand
s = lap2.addVars(N, name="s")

lap2.setObjective(100 * x.sum() + 10 * r.sum() + y.prod(t) + 60 * q.sum() + s.sum(),
                  GRB.MINIMIZE)

# conservation at every node: outflow - inflow = r - d + q - s
node = lap2.addConstrs((y.sum(i, "*") - y.sum("*", i) == r[i] - d[i] + q[i] - s[i]
                        for i in N), name="node")
lap2.addConstr(r.sum() <= B, name="budget")
lap2.addConstrs((r[i] <= B * x[i] for i in N), name="link")
lap2.addConstr(x.sum() <= p, name="cardinality")

lap2.optimize()

print(f"\nopen: {[i for i in N if x[i].X > 0.5]}")
print(f"total cost = {lap2.ObjVal:g}   (same as the reference model)")